# 토스 경진대회 최종 제출 - AP + WLL 최적화

**홍익대 3학년 | 데이터사이언스 | AI 해커톤 준비 중**

---
## 핵심 전략 (슬라이드 기반)
1. **Resampling**: 5-fold CV + **Bootstrap 추정**
2. **Classification**: **p-value 기반 범주형 필터링** → LightGBM
3. **WLL 대응**: `scale_pos_weight=1.0` + **Isotonic Calibration**
4. **AP 최적화**: **OOF 기반 Rank 보정**
5. **Model Assessment**: CV로 **test error 추정**

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 디렉토리 설정
notebook_dir = r'C:\Users\tkdwl\Desktop\토스 경진대회'
os.chdir(notebook_dir)

# 데이터 로드
df_train = pd.read_parquet('./train.parquet')
df_test = pd.read_parquet('./test.parquet')

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

## 1. 결측 처리 (슬라이드: Resampling 전 데이터 정제)

In [2]:
target_col = 'clicked'
id_col = 'ID'
col_drop_threshold = 0.05

# 5% 이상 결측 제거
missing = pd.concat([df_train.isnull().mean(), df_test.isnull().mean()], axis=1).max(axis=1)
high_missing_cols = missing[missing >= col_drop_threshold].index.tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [target_col, id_col]]

df_train = df_train.drop(columns=high_missing_cols)
df_test = df_test.drop(columns=high_missing_cols)

# 수치형/범주형 분리
num_cols = df_train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in [id_col]]

# 수치형 결측 행 제거
df_train = df_train.dropna(subset=num_cols)
df_test = df_test.dropna(subset=num_cols)

# 범주형 최빈값
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df_train[cat_cols] = cat_imputer.fit_transform(df_train[cat_cols])
df_test[cat_cols] = cat_imputer.transform(df_test[cat_cols])

print(f"결측 처리 완료")

## 2. 범주형 p-value 필터링 (Classification 슬라이드)

In [3]:
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

cat_for_selection = ['gender', 'age_group', 'inventory_id', 'day_of_week']
X_cat = df_train[cat_for_selection].astype(str)
y = df_train[target_col]

# One-hot
ohe = OneHotEncoder(sparse=True, handle_unknown='ignore')
X_ohe = ohe.fit_transform(X_cat)
feature_names = ohe.get_feature_names_out(cat_for_selection)

# Logistic + L1 + p-value
X_const = sm.add_constant(X_ohe)
logit = sm.Logit(y, X_const)
result = logit.fit_regularized(method='l1', alpha=0.01, disp=False)

p_vals = result.pvalues[1:]
selected_features = feature_names[p_vals < 0.05]

print(f"선택된 범주형 피처: {len(selected_features)}개")

# 적용
ohe_sel = OneHotEncoder(sparse=False)
ohe_sel.categories_ = ohe.categories_

train_ohe = pd.DataFrame(ohe_sel.fit_transform(X_cat), columns=feature_names)[selected_features]
test_ohe = pd.DataFrame(ohe_sel.transform(df_test[cat_for_selection].astype(str)), columns=feature_names)[selected_features]

for col in selected_features:
    df_train[col] = train_ohe[col].values
    df_test[col] = test_ohe[col].values

df_train = df_train.drop(columns=cat_for_selection)
df_test = df_test.drop(columns=cat_for_selection)

## 3. 피처 엔지니어링

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 시간 피처
df_train['hour'] = pd.to_numeric(df_train['hour'], errors='coerce').fillna(12)
df_test['hour'] = pd.to_numeric(df_test['hour'], errors='coerce').fillna(12)

for df in [df_train, df_test]:
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['is_weekend'] = df['day_of_week'].astype(str).str.contains('5|6').astype(int)

# seq 피처
def process_seq(df, top_n=15):
    df['seq_length'] = df['seq'].str.split(',').str.len().fillna(0)
    df['seq_unique'] = df['seq'].str.split(',').apply(lambda x: len(set(x)) if isinstance(x, str) else 0)
    df['seq_diversity'] = df['seq_unique'] / (df['seq_length'] + 1e-8)
    items = [i.strip() for s in df['seq'].str.split(',') for i in s if isinstance(s, str)]
    top_items = pd.Series(items).value_counts().head(50).index
    for item in top_items[:top_n]:
        df[f'seq_has_{item}'] = df['seq'].str.contains(item, regex=False).fillna(0).astype(int)
    return df

df_train = process_seq(df_train)
df_test = process_seq(df_test)

# 클러스터링
cluster_cols = [c for c in num_cols if c in df_train.columns]
scaler = StandardScaler()
scaled_train = scaler.fit_transform(df_train[cluster_cols])
scaled_test = scaler.transform(df_test[cluster_cols])

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(scaled_train)
df_test['cluster'] = kmeans.predict(scaled_test)

df_train = pd.get_dummies(df_train, columns=['cluster'], prefix='cluster')
df_test = pd.get_dummies(df_test, columns=['cluster'], prefix='cluster')
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# 수치형 스케일링
num_cols_final = [c for c in num_cols if c in df_train.columns]
df_train[num_cols_final] = scaler.fit_transform(df_train[num_cols_final])
df_test[num_cols_final] = scaler.transform(df_test[num_cols_final])

print(f"최종 피처 수: {df_train.shape[1] - 2}")

## 4. 5-fold CV + LightGBM + **WLL 대응**

In [5]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, log_loss
from sklearn.isotonic import IsotonicRegression
import gc

feature_cols = [c for c in df_train.columns if c not in [target_col, id_col]]
X = df_train[feature_cols]
y = df_train[target_col]

# WLL: 50:50 가중 → scale_pos_weight=1.0
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.03,
    'num_leaves': 128,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(df_test))
ap_scores = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_trn, y_trn)
    lgb_valid = lgb.Dataset(X_val, y_val, reference=lgb_train)

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=10000,
        valid_sets=[lgb_valid],
        early_stopping_rounds=300,
        verbose_eval=1000
    )

    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred

    # AP 계산
    ap = average_precision_score(y_val, val_pred)
    ap_scores.append(ap)
    print(f"Fold {fold+1} AP: {ap:.5f}")

    test_preds += model.predict(df_test[feature_cols]) / skf.n_splits

print(f"\nCV AP: {np.mean(ap_scores):.5f} ± {np.std(ap_scores):.5f}")

## 5. 확률 보정 (Isotonic) + WLL 최적화

In [6]:
# Isotonic Calibration (AP + WLL 향상)
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(oof_preds, y)
test_preds_calibrated = iso_reg.predict(test_preds)

# WLL 계산 (50:50 가중)
def weighted_log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    loss_0 = -np.mean((1-y_true) * np.log(1-y_pred))
    loss_1 = -np.mean(y_true * np.log(y_pred))
    return 0.5 * loss_0 + 0.5 * loss_1

wll_oof = weighted_log_loss(y, np.clip(oof_preds, 1e-15, 1-1e-15))
print(f"OOF WLL: {wll_oof:.5f}")

## 6. 최종 제출

In [7]:
submission = pd.DataFrame({
    'ID': df_test[id_col],
    'clicked': test_preds_calibrated
})
submission.to_csv('submission_final_competition.csv', index=False)
print("제출 완료: submission_final_competition.csv")